# F15 — Empaquetado de shards DL sobre Colab (cómputo fuera del PC local)

Política del equipo (13-oct): el cómputo pesado corre en Colab; el repo local
solo recibe los artefactos ligeros ya verificados. Este notebook provisiona
el dato crudo ZZU (~4 GB comprimido), ejecuta `scripts/empaquetar_shards_dl.py`
(int16, shards ~1 GB, splits 80/10/10 por paciente, stats de train) y sube
los shards a Drive para F16–F18.

Las celdas 1–2 (SSH) permiten que el asistente conduzca el resto por SSH
(`ssh colab-f15`); si prefieres la UI de Colab, ejecuta de arriba a abajo.

## 1. Setup SSH (igual que colab_f14)

In [1]:
%%bash
apt-get -qq update >/dev/null && apt-get -qq install -y openssh-server p7zip-full >/dev/null
mkdir -p /run/sshd /root/.ssh && chmod 700 /root/.ssh
sed -i 's/#\?Port .*/Port 22/' /etc/ssh/sshd_config
nohup /usr/sbin/sshd >/dev/null 2>&1 &
echo "sshd arriba"

sshd arriba


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [2]:
PUBKEY = "ssh-ed25519 AAAA... tu@pc"
TS_AUTHKEY = "tskey-auth-..."

open("/root/.ssh/authorized_keys", "w").write(PUBKEY + "\n")
import os; os.chmod("/root/.ssh/authorized_keys", 0o600)
!curl -fsSL https://tailscale.com/install.sh | sh
!tailscale up --hostname colab-f15 --authkey="$TS_AUTHKEY" --ssh=false
!tailscale ip -4

Installing Tailscale for ubuntu noble, using method apt
+ mkdir -p --mode=0755 /usr/share/keyrings
+ curl -fsSL https://pkgs.tailscale.com/stable/ubuntu/noble.noarmor.gpg
+ tee /usr/share/keyrings/tailscale-archive-keyring.gpg
+ chmod 0644 /usr/share/keyrings/tailscale-archive-keyring.gpg
+ curl -fsSL https://pkgs.tailscale.com/stable/ubuntu/noble.tailscale-keyring.list
+ tee /etc/apt/sources.list.d/tailscale.list
# Tailscale packages for ubuntu noble
deb [signed-by=/usr/share/keyrings/tailscale-archive-keyring.gpg] https://pkgs.tailscale.com/stable/ubuntu noble main
+ chmod 0644 /etc/apt/sources.list.d/tailscale.list
+ apt-get update
Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://pkgs.tailscale.com/stable/ubuntu noble InRelease                 
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease     
Hit:4 http://security.ubuntu.com/ubuntu noble-security InRelease               
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease        

## 2. Repo + dependencias + dato crudo ZZU

In [3]:
REPO = "https://codeberg.org/norlek/" \
       "Software_Deteccion_Arritmias-Pediatricas-mediante-PDS_ECG.git"
RAMA = "etapa2-corpus-combinado"

!test -d Software_Deteccion_Arritmias-Pediatricas-mediante-PDS_ECG || git clone --depth 1 "$REPO" -b "$RAMA"
%cd Software_Deteccion_Arritmias-Pediatricas-mediante-PDS_ECG
# sync robusto a clones shallow de otra rama: fetch explícito + checkout de FETCH_HEAD
!git fetch origin "$RAMA" --depth 1
!git checkout -B "$RAMA" FETCH_HEAD
!git log --oneline -1
!pip install -q wfdb pywavelets
import wfdb, pywt, sklearn, pandas, numpy
print('deps OK')

/content/Software_Deteccion_Arritmias-Pediatricas-mediante-PDS_ECG
remote: Total 0 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
From https://codeberg.org/norlek/Software_Deteccion_Arritmias-Pediatricas-mediante-PDS_ECG
 * branch            etapa2-corpus-combinado -> FETCH_HEAD
Reset branch 'etapa2-corpus-combinado'
Your branch is up to date with 'origin/etapa2-corpus-combinado'.
a1a06fe (grafted, HEAD -> etapa2-corpus-combinado, origin/etapa2-corpus-combinado) fix F15: verificador con memmap + stats en chunks — cache en RAM de los 7 shards (~6.6GB) + shard completo float64 (8GB) mataban el kernel Colab por OOM tras bit-exacta 200/200
deps OK


In [4]:
# ZZU pECG multipartito con verificación MD5 (docs/metodologia_zzu_pecg.md §1)
ARCHIVOS = {
    'Child_ecg.z01': ('https://ndownloader.figshare.com/files/53524583', '4782e8c17b478745853db85210ce46aa'),
    'Child_ecg.zip': ('https://ndownloader.figshare.com/files/53523902', 'd7bb207d0f559c5f697bd7e442975718'),
}
import hashlib, pathlib, subprocess
dl = pathlib.Path('/content/descargas'); dl.mkdir(exist_ok=True)
for nombre, (url, md5) in ARCHIVOS.items():
    destino = dl / nombre
    if not destino.exists():
        subprocess.run(['curl', '-L', '-C', '-', '-o', str(destino), url], check=True)
    h = hashlib.md5(destino.read_bytes()).hexdigest()
    assert h == md5, f'{nombre}: MD5 {h} != {md5}'
    print(nombre, 'MD5 OK')
data = pathlib.Path('data/zzu_pecg'); data.mkdir(parents=True, exist_ok=True)
if not (data / 'Child_ecg').exists():
    subprocess.run(['7z', 'x', f'-o{data}', str(dl / 'Child_ecg.zip')], check=True)
n_hea = len(list((data / 'Child_ecg').rglob('*.hea')))
assert n_hea == 14190, n_hea
print(f'Child_ecg: {n_hea} registros')

Child_ecg.z01 MD5 OK
Child_ecg.zip MD5 OK
Child_ecg: 14190 registros


## 3. F15 — empaquetado + verificación (~40–60 min en runtime CPU)

In [5]:
!python scripts/empaquetar_shards_dl.py
import os
assert os.path.exists('results/dl/verificacion_shards.txt'), \
    'verificación incompleta: corre la celda de recuperación (--verificar)'
!cat results/dl/verificacion_shards.txt
!cat results/dl/normalizacion.json

58178 ventanas a empaquetar (con Otro)
7 shards de ~8333 ventanas (0 MB/ventana int16)
    500/13436 registros
    1000/13436 registros
    1500/13436 registros
    shard_000.npy: (8333, 12, 5000) (1.00 GB)
    2000/13436 registros
    2500/13436 registros
    3000/13436 registros
    3500/13436 registros
    shard_001.npy: (8333, 12, 5000) (1.00 GB)
    4000/13436 registros
    4500/13436 registros
    5000/13436 registros
    5500/13436 registros
    shard_002.npy: (8333, 12, 5000) (1.00 GB)
    6000/13436 registros
    6500/13436 registros
    7000/13436 registros
    7500/13436 registros
    shard_003.npy: (8333, 12, 5000) (1.00 GB)
    8000/13436 registros
    8500/13436 registros
    9000/13436 registros
    9500/13436 registros
    shard_004.npy: (8333, 12, 5000) (1.00 GB)
    10000/13436 registros
    10500/13436 registros
    11000/13436 registros
    shard_005.npy: (8333, 12, 5000) (1.00 GB)
    11500/13436 registros
    12000/13436 registros
    12500/13436 registros
    130

## 4. Persistencia: shards a Drive, manifiesto a git

In [ ]:
import os
LIGEROS = ['results/dl/manifiesto.csv', 'results/dl/splits.csv',
           'results/dl/normalizacion.json', 'results/dl/verificacion_shards.txt']
faltan = [f for f in LIGEROS if not os.path.exists(f)]
assert not faltan, f'faltan artefactos (corre la recuperación con --verificar): {faltan}'

from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/arritmias_dl/shards
!cp --update=none results/dl/shards/*.npy /content/drive/MyDrive/arritmias_dl/shards/
!cp results/dl/manifiesto.csv results/dl/splits.csv results/dl/normalizacion.json results/dl/verificacion_shards.txt /content/drive/MyDrive/arritmias_dl/
!ls -lh /content/drive/MyDrive/arritmias_dl/shards | tail -3
# Artefactos ligeros al repo (rama etapa2-corpus-combinado):
!git config user.email "gustavo.castro14159@protonmail.com" && git config user.name "norlek"
!git add results/dl/manifiesto.csv results/dl/splits.csv results/dl/normalizacion.json results/dl/verificacion_shards.txt
!git commit -m "F15: shards DL empaquetados en Colab (verificacion VERDE)"
!git push origin etapa2-corpus-combinado

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cp: warning: behavior of -n is non-portable and may change in future; use --update=none instead
total 6.6G
-rw------- 1 root root 954M Oct 13 21:42 shard_000.npy
-rw------- 1 root root 954M Oct 13 21:42 shard_001.npy
-rw------- 1 root root 954M Oct 13 21:42 shard_002.npy
-rw------- 1 root root 954M Oct 13 21:42 shard_003.npy
-rw------- 1 root root 954M Oct 13 21:42 shard_004.npy
-rw------- 1 root root 954M Oct 13 21:43 shard_005.npy
-rw------- 1 root root 937M Oct 13 21:43 shard_006.npy
[etapa2-corpus-combinado 029cd01] F15: shards DL empaquetados en Colab (verificacion VERDE)
 2 files changed, 52 insertions(+)
 create mode 100644 results/dl/normalizacion.json
 create mode 100644 results/dl/verificacion_shards.txt
fatal: could not read Username for 'https://codeberg.org': No such device or address


In [8]:
REPO = "https://codeberg.org/norlek/" \
       "Software_Deteccion_Arritmias-Pediatricas-mediante-PDS_ECG.git"

%cd /content
!rm -rf Software_Deteccion_Arritmias-Pediatricas-mediante-PDS_ECG
!git clone --depth 1 "$REPO" -b etapa2-corpus-combinado
%cd Software_Deteccion_Arritmias-Pediatricas-mediante-PDS_ECG
!pip install -q wfdb pywavelets

/content
Cloning into 'Software_Deteccion_Arritmias-Pediatricas-mediante-PDS_ECG'...
remote: Enumerating objects: 105, done.
remote: Counting objects: 100% (105/105), done.
remote: Compressing objects: 100% (101/101), done.
remote: Total 105 (delta 1), reused 44 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (105/105), 19.42 MiB | 6.42 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/Software_Deteccion_Arritmias-Pediatricas-mediante-PDS_ECG


In [ ]:
import pathlib, hashlib, subprocess
dl = pathlib.Path('/content/descargas'); dl.mkdir(exist_ok=True)
FILES = {'Child_ecg.z01':
('https://ndownloader.figshare.com/files/53524583','4782e8c17b478745853db85210ce46aa'),
         'Child_ecg.zip':
('https://ndownloader.figshare.com/files/53523902','d7bb207d0f559c5f697bd7e442975718')}
data = pathlib.Path('data/zzu_pecg')
if not (data/'Child_ecg').exists():
    for n,(u,md5) in FILES.items():
        f = dl/n
        if not f.exists(): subprocess.run(['curl','-L','-C','-','-o',str(f),u],check=True)
        assert hashlib.md5(f.read_bytes()).hexdigest()==md5, n
    data.mkdir(parents=True, exist_ok=True)
    subprocess.run(['7z','x',f'-o{data}',str(dl/'Child_ecg.zip')],check=True)
print('registros:', len(list((data/'Child_ecg').rglob('*.hea'))))

from google.colab import drive
drive.mount('/content/drive')
!mkdir -p results/dl/shards
!cp -n /content/drive/MyDrive/arritmias_dl/shards/*.npy results/dl/shards/
!ls -lh results/dl/shards | tail -3
!python scripts/empaquetar_shards_dl.py --verificar
!cat results/dl/verificacion_shards.txt && cat results/dl/normalizacion.json